# FrameGuard — thử nhanh trên Kaggle

Mục tiêu: trả lời nhanh 3 câu hỏi trước khi đầu tư tiếp.
1. **Chạy được không?** vLLM + Qwen3.5-2B (fp16 trên T4) có gọi tool đúng định dạng, có thinking.
2. **Có giống baseline công bố không?** Chế độ `none` (không Gate) trên 12 task bank so với kết quả SOPBench đã công bố.
3. **Có tín hiệu tích cực không?** Gate + phản hồi `alignment` có giúp model hoàn thành các task mà baseline thất bại không, so với `bare` (chỉ báo bị chặn).

**Thiết lập trước khi chạy** (cột phải → *Session options*):
- *Accelerator*: **GPU T4 x2**
- *Internet*: **On** (cần tài khoản đã xác minh số điện thoại)
- Bấm **Save Version → Save & Run All (Commit)** để chạy nền; có thể tắt trình duyệt.

Tập 12 task là **tập làm giàu**: 8 task mà Qwen3.5-2B đã thất bại vì bỏ bước kiểm tra + 4 task đã pass.
Không dùng để ước lượng pass rate, chỉ để xem có tín hiệu hay không.

In [ ]:
# ===== Cấu hình =====
MODEL    = "qwen3.5-2b"              # hoặc "gemma-4-e2b-it" (lưu ý: Gemma có thể không ổn định với fp16)
DOMAIN   = "bank"
MODES    = "none,bare,alignment"     # thêm reason,next_action,template nếu còn thời gian
TASK_SET = "smoke"                   # "smoke" (12 task) | "clean" (toàn bộ task sạch) | "8,67,81"
import os
os.environ["VLLM_DTYPE"] = "float16"          # T4 không hỗ trợ bf16 (SOPBench gốc dùng bf16)
os.environ["QWEN35_THINKING"] = "1"           # bản công bố chạy CÓ thinking
os.environ["GEMMA4_THINKING"] = "1"
# os.environ["VLLM_MAX_MODEL_LEN"] = "16384"  # bật nếu bị thiếu bộ nhớ (mặc định SOPBench: 32000)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working/frameguard"

In [ ]:
# Dừng sớm nếu phiên không có GPU (tránh chạy CPU 15 phút rồi mới lỗi)
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv"], capture_output=True, text=True) \
    if __import__("shutil").which("nvidia-smi") else None
if r is None or r.returncode != 0:
    raise SystemExit("KHÔNG CÓ GPU. Vào Session options -> Accelerator -> GPU T4 x2 "
                     "(cần xác minh số điện thoại; kiểm tra quota GPU còn lại ở kaggle.com/settings), rồi commit lại.")
print(r.stdout)

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/frameguard/external && cd /kaggle/working/frameguard/external
if [ ! -d SOPBench ]; then
  # bỏ thư mục output/ (1.9GB trajectory công bố) cho nhẹ
  git clone -q --filter=blob:none --no-checkout https://github.com/Leezekun/SOPBench.git
  cd SOPBench
  git sparse-checkout set --no-cone '/*' '!/output/'
  git checkout -q 8790ed7406acbd22270f90bd1c4038de3fc5db8f
fi
echo "SOPBench @ $(git -C /kaggle/working/frameguard/external/SOPBench rev-parse --short HEAD)"

In [ ]:
%%bash
# vLLM 0.25.0 là bản SOPBench dùng cho Qwen3.5 / Gemma-4 (cài mất khoảng 5–10 phút)
pip install -q "vllm==0.25.0" 2>&1 | tail -3
pip install -q "openai>=1.71.0" "anthropic>=0.49.0" python-dotenv "pydantic>=2.9.2" termcolor colorama tqdm 2>&1 | tail -3
python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## Mã FrameGuard (nhúng sẵn, sinh từ repo)

In [ ]:
!mkdir -p /kaggle/working/frameguard/analysis /kaggle/working/frameguard/frameguard /kaggle/working/frameguard/adapters/sopbench

In [ ]:
%%writefile /kaggle/working/frameguard/analysis/declare_rules.py
"""Chuyển luật SOPBench sang DECLARE / MP-DECLARE.

Nguồn luật (đã đối chiếu với env/evaluator.py):
  * Luồng điều khiển: DAG AND/OR ở mức tool. user_goal dùng task["directed_action_graph"];
    action khác dùng env.helpers.dfsgather_ifg_func(..., "full") — giống oracle.
  * Điều kiện dữ liệu: cây vị từ task["constraints"] (user_goal) và get_default_dep_full (action khác),
    đánh giá bằng phát lại trên hệ thống strict của SOPBench.

Ánh xạ:
  node tool X, con là tool Y          -> Precedence(Y, X)            (+ tương quan tham số)
  con "and"                           -> nhiều ràng buộc độc lập
  con "or", mọi nhánh là tool         -> BranchedPrecedence({Y..}, X)
  con "or", có nhánh phức hợp         -> ChoicePrecedence (DNF; ngoài DECLARE chuẩn)
  tool lồng nhau (Chain)              -> ràng buộc riêng trên chính Y
  vị từ dữ liệu của X                 -> MP-DECLARE activation condition
  action_should_succeed               -> Existence(goal) | Absence(goal thành công)
"""
from __future__ import annotations
import copy, os, sys
from dataclasses import dataclass, field

SOPBENCH = os.environ.get("SOPBENCH", os.path.join(os.path.dirname(__file__), "..", "external", "SOPBench"))
sys.path.insert(0, os.path.abspath(SOPBENCH))

from env.variables import domain_keys, domain_assistant_keys  # noqa: E402
from env.task import get_default_dep_full                     # noqa: E402
from env.helpers import dfsgather_ifg_func, get_ifg_connections_invnodes, get_action_parameters  # noqa: E402

OPS = ("single", "and", "or", "chain", "gate")


# ---------------------------------------------------------------- công thức tiền điều kiện
@dataclass
class Act:
    name: str
    params: dict          # tham số của Y -> tên biến dùng chung với X
    sub: object = None    # công thức tiền điều kiện của chính Y trong DAG (Chain)


@dataclass
class Op:
    kind: str             # "and" | "or"
    parts: list


def _compile(nodes, conns, i):
    node = nodes[i]
    if isinstance(node, str):
        return Op(node, [_compile(nodes, conns, c) for c in sorted(conns[i])])
    name, params = node
    kids = [_compile(nodes, conns, c) for c in sorted(conns[i])]
    sub = None if not kids else kids[0] if len(kids) == 1 else Op("and", kids)
    return Act(name, dict(params or {}), sub)


def graph_formula(graph):
    """DAG của SOPBench -> (Act gốc, inv_nodes). Gốc là action được kích hoạt."""
    conns, inv = get_ifg_connections_invnodes(graph)
    return _compile(graph["nodes"], conns, 0), inv


def graph_node_formula(graph, action):
    """Công thức tiền điều kiện của `action` theo node đầu tiên của nó trong DAG (cách oracle làm)."""
    conns, inv = get_ifg_connections_invnodes(graph)
    if action not in inv:
        return None
    return _compile(graph["nodes"], conns, inv[action])


# ---------------------------------------------------------------- liệt kê template DECLARE
@dataclass(frozen=True)
class Constraint:
    template: str         # Precedence | BranchedPrecedence | ChoicePrecedence | Existence | Absence | DataCondition
    target: str
    sources: tuple = ()   # tên tool (hoặc nhóm) phải xảy ra trước
    detail: str = ""

    def __str__(self):
        s = "|".join(self.sources)
        return f"{self.template}({s + ', ' if s else ''}{self.target}){' [' + self.detail + ']' if self.detail else ''}"


def _acts(f):
    if isinstance(f, Act):
        return [f]
    return [a for p in f.parts for a in _acts(p)]


def _dnf_str(f):
    if isinstance(f, Act):
        return f.name
    j = " & " if f.kind == "and" else " | "
    return "(" + j.join(_dnf_str(p) for p in f.parts) + ")"


def constraints_of(act: Act, out=None):
    """Sinh các ràng buộc DECLARE từ một node tool và các tiền điều kiện lồng nhau của nó."""
    out = [] if out is None else out
    if act.sub is None:
        return out
    stack = [act.sub]
    while stack:
        f = stack.pop()
        if isinstance(f, Act):
            out.append(Constraint("Precedence", act.name, (f.name,)))
        elif f.kind == "and":
            stack.extend(f.parts)
        elif all(isinstance(p, Act) for p in f.parts):
            out.append(Constraint("BranchedPrecedence", act.name, tuple(p.name for p in f.parts)))
        else:
            out.append(Constraint("ChoicePrecedence", act.name, (_dnf_str(f),)))
    for y in _acts(act.sub):
        constraints_of(y, out)
    return out


# ---------------------------------------------------------------- vị từ dữ liệu
def pred_leaves(dep):
    if not dep:
        return []
    if dep[0] == "single":
        return [dep]
    return [x for d in dep[1] for x in pred_leaves(d)]


def pred_name(leaf):
    return leaf[1].replace("not ", "")


def verifying_tools(domain, pred):
    """Các tool cho agent 'thấy' giá trị của vị từ (để phân biệt 'chưa kiểm tra' và 'kiểm tra rồi vẫn làm')."""
    da = domain_assistant_keys[domain]
    tools = set()
    if pred in da.constraint_links:
        cl = da.constraint_links[pred]
        tools.add(cl[0] if isinstance(cl, tuple) else cl)
    if pred in getattr(da, "constraint_processes", {}) and da.constraint_processes[pred]:
        tools |= {pred_name(l) for l in pred_leaves(da.constraint_processes[pred])}
    if pred in da.action_descriptions:
        tools.add(pred)
    return tools


def pred_kind(domain, pred):
    """state: được Precedence trên tool liên kết bao phủ; data: điều kiện dữ liệu (MP-DECLARE)."""
    return "state" if pred in domain_assistant_keys[domain].constraint_links else "data"


# ---------------------------------------------------------------- load_rules
@dataclass
class RuleSet:
    domain: str
    task: dict
    goal: str
    should_succeed: bool
    goal_graph: dict
    dep_full: dict                      # cây vị từ cho mọi action (goal đã thay bằng task["constraints"])
    action_params: dict
    _graph_cache: dict = field(default_factory=dict)

    def graph_for(self, action):
        """DAG dùng để kiểm Precedence cho `action` (task graph nếu action có trong đó, như oracle)."""
        _, inv = get_ifg_connections_invnodes(self.goal_graph)
        if action in inv:
            return self.goal_graph
        if action not in self._graph_cache:
            ds = domain_keys[self.domain + "_strict"]().evaluation_get_domain_system()
            self._graph_cache[action] = dfsgather_ifg_func(ds, domain_assistant_keys[self.domain], action, "full", False)
        return self._graph_cache[action]

    def precondition(self, action):
        g = self.graph_for(action)
        return graph_node_formula(g, action) if g else None

    def declare(self):
        """Toàn bộ mô hình DECLARE của task (để báo cáo / sinh phản hồi)."""
        root, _ = graph_formula(self.goal_graph)
        cs = constraints_of(root)
        for leaf in pred_leaves(self.task["constraints"]):
            p = pred_name(leaf)
            if pred_kind(self.domain, p) == "data":
                cs.append(Constraint("DataCondition", self.goal, (), leaf[1]))
        cs.append(Constraint("Existence" if self.should_succeed else "Absence", self.goal))
        return list(dict.fromkeys(cs))


def load_rules(domain: str, task: dict) -> RuleSet:
    dep_full = get_default_dep_full(domain, "full")
    dep_full[task["user_goal"]] = task["constraints"]
    ds = domain_keys[domain + "_strict"]().evaluation_get_domain_system()
    return RuleSet(domain, task, task["user_goal"], task["action_should_succeed"],
                   copy.deepcopy(task["directed_action_graph"]), dep_full,
                   get_action_parameters(ds, domain_assistant_keys[domain]))


def new_strict_system(rs: RuleSet):
    """Hệ thống strict của SOPBench với DB ban đầu của task (dùng để đánh giá điều kiện dữ liệu khi phát lại)."""
    return domain_keys[rs.domain + "_strict"](
        copy.deepcopy(rs.task["initial_database"]),
        domain_assistant_keys[rs.domain].action_innate_dependencies,
        copy.deepcopy(rs.dep_full), rs.task["constraint_parameters"])


if __name__ == "__main__":
    import json
    dom = sys.argv[1] if len(sys.argv) > 1 else "bank"
    data = json.load(open(os.path.join(SOPBENCH, "data", f"{dom}_tasks.json")))
    for goal, tasks in list(data.items())[:3]:
        rs = load_rules(dom, tasks[0])
        print(f"\n# {goal} (should_succeed={rs.should_succeed})")
        for c in rs.declare():
            print("  ", c)

In [ ]:
%%writefile /kaggle/working/frameguard/analysis/conformance.py
"""Conformance checking DECLARE/MP-DECLARE trên trajectory công bố của SOPBench (giai đoạn 1).

Dùng: .venv/bin/python analysis/conformance.py [domain ...]
Ra:   analysis/out/conformance_<domain>.json  (vi phạm từng trace) + bảng tóm tắt + độ khớp với oracle.
"""
from __future__ import annotations
import ast, collections, copy, glob, json, os, sys

sys.path.insert(0, os.path.dirname(__file__))
from declare_rules import (SOPBENCH, Act, Op, load_rules, new_strict_system, pred_leaves, pred_name,  # noqa: E402
                           pred_kind, verifying_tools, domain_assistant_keys)
from failure_taxonomy import SMALL, STRONG, SUFFIX  # noqa: E402

OUT = os.path.join(os.path.dirname(__file__), "out")


# ---------------------------------------------------------------- trích tool call giống run_evaluation.py
def try_eval(s):
    if not isinstance(s, str):
        return s
    try:
        return ast.literal_eval(s)
    except Exception:
        return s


def extract_calls(interaction):
    calls = []
    for i in range(len(interaction) - 1):
        tcs = [tc for tc in (interaction[i].get("tool_calls") or [])
               if tc["function"]["name"].lower() not in ("n/a", "na", "none", "null")]
        if tcs and "tool_name" in interaction[i + 1]:
            args = try_eval(tcs[0]["function"]["arguments"])
            calls.append({"name": interaction[i + 1]["tool_name"],
                          "args": args if isinstance(args, dict) else {},
                          "result": try_eval(interaction[i + 1]["content"])})
    return calls


def succeeded(res):
    if isinstance(res, bool):
        return res
    if isinstance(res, (tuple, list)) and res and isinstance(res[0], bool):
        return res[0]
    return False


def norm(v):
    return json.dumps(v, sort_keys=True) if isinstance(v, (list, dict)) else v


# ---------------------------------------------------------------- kiểm Precedence (ngữ nghĩa giống oracle)
class History:
    """Các lời gọi 'hợp lệ' (thỏa tiền điều kiện lúc gọi) và mọi lời gọi, theo tên tool."""
    def __init__(self):
        self.conformant = collections.defaultdict(list)   # name -> [args]
        self.any = collections.defaultdict(list)

    def match(self, act: Act, binding: dict, pool):
        exp = {k: binding.get(v) for k, v in act.params.items()}
        for args in pool.get(act.name, []):
            if all(exp[k] is None or norm(args.get(k)) == norm(exp[k]) for k in exp):
                return True
        return False


def unsatisfied(f, binding, hist):
    """Trả về danh sách vi phạm tối thiểu của công thức tiền điều kiện f (rỗng = thỏa)."""
    if f is None:
        return []
    if isinstance(f, Act):
        if hist.match(f, binding, hist.conformant):
            return []
        if hist.match(f, binding, hist.any):
            kind = "unconformant"          # Y đã gọi nhưng chính Y vi phạm tiền điều kiện (Chain)
        elif f.name in hist.any:
            kind = "param_mismatch"        # Y đã gọi nhưng với tham số khác
        else:
            kind = "missing"
        return [("Precedence", (f.name,), kind)]
    if f.kind == "and":
        return [v for p in f.parts for v in unsatisfied(p, binding, hist)]
    # or
    sub = [unsatisfied(p, binding, hist) for p in f.parts]
    if any(not s for s in sub):
        return []
    names = tuple(sorted({a.name for p in f.parts for a in _acts(p)}))
    tpl = "BranchedPrecedence" if all(isinstance(p, Act) for p in f.parts) else "ChoicePrecedence"
    kinds = {k for s in sub for (_, _, k) in s}
    return [(tpl, names, "missing" if kinds == {"missing"} else "partial")]


def _acts(f):
    return [f] if isinstance(f, Act) else [a for p in f.parts for a in _acts(p)]


def add_implied(rs, name, args, hist):
    """Gọi X hợp lệ ngụ ý các phụ thuộc bẩm sinh của X đã được kiểm (oracle cũng làm vậy)."""
    da = domain_assistant_keys[rs.domain]
    dep = da.action_innate_dependencies.get(name)
    # Oracle (dfsgather_allfunccalled_indepperm) chỉ xét nhánh đầu của and/or — giữ nguyên để đối chiếu.
    while dep and dep[0] != "single":
        dep = dep[1][0] if dep[1] else None
    if not dep:
        return
    p = pred_name(dep)
    if p in rs.action_params:
        leaves = [("single", p, dep[2])]
    elif p in getattr(da, "constraint_processes", {}):
        leaves = [(l[0], pred_name(l), {k: (dep[2] or {}).get(v, v) for k, v in (l[2] or {}).items()})
                  for l in pred_leaves(da.constraint_processes[p])]
    else:
        return
    for _, tool, pm in leaves:
        if tool in rs.action_params:
            mapped = {k: args.get(v) for k, v in (pm or {}).items()}
            hist.conformant[tool].append(mapped)
            hist.any[tool].append(mapped)


# ---------------------------------------------------------------- phát lại một trace
def check_trace(rs, interaction):
    calls = extract_calls(interaction)
    strict = new_strict_system(rs)
    dep_eval = strict.evaluation_get_dependency_evaluator()
    hist = History()
    seen = set()                                   # tool đã gọi (để biết agent đã 'thấy' vị từ chưa)
    viol = []
    tool_error = False
    strict_rejected = False
    goal_success = False
    for step, c in enumerate(calls):
        name, args, res = c["name"], c["args"], c["result"]
        if name not in rs.action_params:
            seen.add(name)
            continue
        # --- control-flow: Precedence / Branched / Choice
        f = rs.precondition(name)
        node_params = f.params if isinstance(f, Act) else {}
        binding = {v: args.get(k) for k, v in node_params.items()}
        miss = unsatisfied(f.sub if isinstance(f, Act) else None, binding, hist)
        for tpl, src, kind in miss:
            viol.append({"step": step, "template": tpl, "target": name, "sources": list(src), "kind": kind,
                         "is_goal": name == rs.goal})
        # --- data: MP-DECLARE activation condition, đánh giá trên hệ thống strict tại thời điểm gọi
        try:
            allowed = dep_eval.process(method_str=name, **args)
        except Exception:
            allowed, tool_error = None, True
        if allowed is False and res is not False and not (isinstance(res, (tuple, list)) and res and res[0] is False):
            strict_rejected = True                 # cùng định nghĩa với constraint_not_violated của oracle
        if allowed is False and succeeded(res):
            for leaf in pred_leaves(rs.dep_full.get(name)):
                try:
                    ok = dep_eval._process(leaf, **args)
                except Exception:
                    continue
                if ok:
                    continue
                p = pred_name(leaf)
                if pred_kind(rs.domain, p) == "state":
                    continue                       # đã được Precedence trên tool liên kết bao phủ
                observed = bool(verifying_tools(rs.domain, p) & seen)
                viol.append({"step": step, "template": "DataCondition", "target": name, "sources": [leaf[1]],
                             "kind": "checked_ignored" if observed else "unchecked", "is_goal": name == rs.goal})
        # --- cập nhật trạng thái
        if not miss:
            hist.conformant[name].append(args)
            add_implied(rs, name, args, hist)
        hist.any[name].append(args)
        seen.add(name)
        if hasattr(strict, name):                  # tool chỉ có ở hệ thống thường: oracle bỏ qua (status 1)
            try:
                getattr(strict, name)(**args)
            except Exception:
                tool_error = True
        if name == rs.goal and succeeded(res):
            goal_success = True
    if rs.should_succeed and not goal_success:
        attempted = any(c["name"] == rs.goal for c in calls)
        viol.append({"step": len(calls), "template": "Existence", "target": rs.goal, "sources": [],
                     "kind": "attempted_failed" if attempted else "never_attempted", "is_goal": True})
    if not rs.should_succeed and goal_success:
        if not any(v["template"] == "DataCondition" and v["is_goal"] for v in viol):
            viol.append({"step": len(calls), "template": "Absence", "target": rs.goal, "sources": [],
                         "kind": "executed", "is_goal": True})
    return {"n_calls": len(calls), "violations": viol, "tool_error": tool_error, "strict_rejected": strict_rejected}


# ---------------------------------------------------------------- chạy theo domain
# Goal có tool không được khai báo schema trong bank_assistant.py → agent không thể gọi (lỗi benchmark).
BROKEN_GOALS = {"bank": {"cancel_credit_card", "pay_bill_with_credit_card"}}


def unsolvable(domain):
    ok, n, goals = collections.Counter(), None, {}
    for p in glob.glob(os.path.join(SOPBENCH, "output", domain, "ast_*-tool_full-shuffle_False.json")):
        d = json.load(open(p))
        if not all(t.get("evaluations") for t in d) or (n is not None and len(d) != n):
            continue
        n = len(d)
        for i, t in enumerate(d):
            ok[i] += t["evaluations"][0]["success"]
            goals[i] = t["task"]["user_goal"]
    return {i for i in range(n or 0) if ok[i] == 0 or goals[i] in BROKEN_GOALS.get(domain, ())}


def first_label(v):
    if not v:
        return "conform"
    x = min(v, key=lambda e: e["step"])
    return f'{x["template"]}:{x["kind"]}' + ("" if x["is_goal"] or x["template"] in ("Existence", "Absence") else "@aux")


def run(domain):
    bad = unsolvable(domain)
    rows, per_trace = {}, []
    agree = collections.Counter()
    for key in SMALL + STRONG:
        p = os.path.join(SOPBENCH, "output", domain, f"ast_{key}{SUFFIX}")
        if not os.path.exists(p):
            continue
        data = json.load(open(p))
        if not all(t.get("evaluations") for t in data):
            continue
        cnt_first, cnt_any, n, ok = collections.Counter(), collections.Counter(), 0, 0
        for i, t in enumerate(data):
            if i in bad:
                continue
            rs = load_rules(domain, t["task"])
            ev = t["evaluations"][0]
            r = check_trace(rs, t["interactions"][0]["interaction"])
            v = r["violations"]
            n += 1
            ok += ev["success"]
            prec = any(x["template"].endswith("Precedence") for x in v)
            agree[("dirgraph", prec == (not ev["dirgraph_satisfied"]))] += 1
            agree[("constraint", r["strict_rejected"] == (not ev["constraint_not_violated"]))] += 1
            agree[("success", (not v and not r["tool_error"]) == ev["success"])] += 1
            if not ev["success"]:
                cnt_first[first_label(v)] += 1
                for tpl in {x["template"] + ":" + x["kind"] for x in v}:
                    cnt_any[tpl] += 1
            per_trace.append({"model": key, "task": i, "goal": rs.goal, "should_succeed": rs.should_succeed,
                              "oracle_success": ev["success"], **r})
        rows[key] = {"n": n, "pass": ok, "first": dict(cnt_first), "any": dict(cnt_any)}
    return rows, per_trace, agree, bad


def report(domain, rows, agree, bad):
    print(f"\n===== {domain}  (loại {len(bad)} task không giải được) =====")
    for k in ("dirgraph", "constraint", "success"):
        a, b = agree[(k, True)], agree[(k, False)]
        print(f"  khớp oracle [{k:10s}]: {a}/{a + b} = {a / max(1, a + b):.1%}")
    labels = sorted({l for r in rows.values() for l in r["first"]})
    print("  Vi phạm ĐẦU TIÊN trong trace thất bại (theo template:kiểu):")
    print(f"  {'model':34s} pass  " + " ".join(f"{l[:22]:>22s}" for l in labels))
    for k, r in rows.items():
        print(f"  {k:34s} {r['pass'] / r['n']:5.1%} " + " ".join(f"{r['first'].get(l, 0):22d}" for l in labels))


if __name__ == "__main__":
    os.makedirs(OUT, exist_ok=True)
    domains = sys.argv[1:] or sorted(os.listdir(os.path.join(SOPBENCH, "output")))
    for d in domains:
        rows, per_trace, agree, bad = run(d)
        report(d, rows, agree, bad)
        json.dump({"rows": rows, "excluded_tasks": sorted(bad),
                   "agreement": {f"{k}:{v}": c for (k, v), c in agree.items()}, "traces": per_trace},
                  open(os.path.join(OUT, f"conformance_{d}.json"), "w"), indent=1, ensure_ascii=False, default=str)

In [ ]:
%%writefile /kaggle/working/frameguard/analysis/failure_taxonomy.py
"""Phân loại lỗi trên trajectory đã công bố của SOPBench (không chạy model).

Dùng: python3 -I analysis/failure_taxonomy.py <SOPBench/output> [domain ...]
Kết quả: in bảng ra stdout và ghi analysis/out/taxonomy_<domain>.json
"""
import json, os, sys, collections

SMALL = ["qwen2.5-7b-instruct-mode_react", "llama3.1-8b-instruct-mode_react",
         "qwen3.5-2b-mode_fc", "qwen3.5-4b-mode_fc",
         "gemma-4-e2b-it-mode_fc", "gemma-4-e4b-it-mode_fc"]
STRONG = ["gpt-4o-mode_fc", "o4-mini-high-mode_fc", "gpt-5-mode_fc"]
SUFFIX = "-dep_full-fmt_structured-tool_full-shuffle_False.json"
OPS = ("single", "and", "or", "chain", "gate")


def constraint_funcs(c, acc=None):
    acc = set() if acc is None else acc
    if isinstance(c, list) and c and c[0] in OPS:
        if c[0] == "single":
            acc.add(c[1].replace("not ", ""))
        else:
            for x in c[1]:
                constraint_funcs(x, acc)
    return acc


def structure(c):
    if not c:
        return "none"
    kinds = set()
    def walk(x):
        if isinstance(x, list) and x and x[0] in OPS:
            kinds.add(x[0])
            if x[0] != "single":
                for y in x[1]:
                    walk(y)
    walk(c)
    kinds.discard("single")
    return "+".join(sorted(kinds)) or "single"


def tool_calls(interaction):
    """Danh sách (tên tool, kết quả) theo thứ tự, lấy từ log."""
    calls, pending = [], {}
    for m in interaction:
        for tc in (m.get("tool_calls") or []):
            if m.get("sender") == "user":
                continue
            pending[tc.get("id")] = tc["function"]["name"]
        if "tool_call_id" in m:
            calls.append((m.get("tool_name") or pending.get(m["tool_call_id"]), m.get("content")))
    return calls


def degenerate(interaction):
    """Lượt assistant không gọi tool và không có văn bản hợp lệ (ReAct bị vỡ định dạng)."""
    bad = 0
    for m in interaction:
        if m.get("sender", "").endswith("assistant") and not m.get("tool_calls"):
            t = (m.get("content") or "").strip()
            if t.startswith("Thought") and "Action" not in t and len(t) < 2000:
                bad += 1
    return bad


def label(task, ev, interaction):
    """Nhãn lỗi chính (loại trừ lẫn nhau, theo thứ tự ưu tiên) cho trajectory thất bại."""
    goal, should = task["user_goal"], task["action_should_succeed"]
    checks = constraint_funcs(task["constraints"]) - {goal}
    calls = tool_calls(interaction)
    names = [n for n, _ in calls]
    target_idx = [i for i, n in enumerate(names) if n == goal]
    before = set(names[: target_idx[0]]) if target_idx else set(names)
    checked_any = bool(checks & before)
    checked_all = checks <= before
    repeated_target = len(target_idx) > 1

    if not ev["no_tool_call_error"]:
        return "F_format_tool_error"
    if not names and degenerate(interaction):
        return "F_format_collapse_no_call"
    if not should and ev["action_successfully_called"]:
        # làm hành động dù điều kiện không đạt
        if checked_all:
            return "C_checked_but_ignored"
        if checked_any:
            return "C_partial_check_then_acted"
        return "A_no_check_acted"
    if should and not ev["action_successfully_called"]:
        if goal in names:
            return "R_called_but_failed"          # gọi nhưng env trả False (thường sai tham số)
        if checked_all:
            return "R_false_refusal_after_check"  # đủ điều kiện, đã kiểm tra, vẫn không làm
        if names:
            return "R_gave_up_partial"
        return "R_no_call_at_all"
    if not ev["dirgraph_satisfied"]:
        return "A_skipped_precondition" if target_idx and not checked_all else "A_order_other_action"
    if not ev["constraint_not_violated"]:
        return "V_other_action_violated"
    if not ev["database_match"]:
        return "D_database_mismatch"
    return "X_other"


def analyze(out_dir, domain):
    rows, per_task = {}, collections.defaultdict(dict)
    for key in SMALL + STRONG:
        p = os.path.join(out_dir, domain, f"ast_{key}{SUFFIX}")
        if not os.path.exists(p):
            continue
        data = json.load(open(p))
        cnt, n, ok = collections.Counter(), 0, 0
        if "evaluations" not in data[0]:
            continue
        for i, t in enumerate(data):
            task, ev, inter = t["task"], t["evaluations"][0], t["interactions"][0]["interaction"]
            n += 1
            succ = ev["success"]
            ok += succ
            lab = "OK" if succ else label(task, ev, inter)
            cnt[lab] += 1
            per_task[i][key] = lab
            per_task[i]["_meta"] = (task["user_goal"], task["action_should_succeed"], structure(task["constraints"]),
                                     ev["num_constraints_expanded"])
        rows[key] = {"n": n, "pass": ok, "labels": dict(cnt)}
    return rows, per_task


def report(domain, rows, per_task):
    print(f"\n===== {domain} =====")
    labels = sorted({l for r in rows.values() for l in r["labels"]} - {"OK"})
    print(f"{'model':40s} pass  " + " ".join(f"{l[:14]:>14s}" for l in labels))
    for k, r in rows.items():
        print(f"{k:40s} {r['pass']/r['n']:5.1%} " + " ".join(f"{r['labels'].get(l,0):14d}" for l in labels))
    small = [k for k in SMALL if k in rows]
    strong = [k for k in STRONG if k in rows]
    # tỷ lệ pass của model nhỏ theo đặc điểm task
    for name, fn in [("should_succeed", lambda m: m[1]), ("structure", lambda m: m[2]),
                     ("n_constraints", lambda m: min(m[3], 5)), ("goal", lambda m: m[0])]:
        agg = collections.defaultdict(lambda: [0, 0])
        for t in per_task.values():
            g = fn(t["_meta"])
            for k in small:
                agg[g][0] += t.get(k) == "OK"
                agg[g][1] += 1
        print(f"-- small-model pass by {name}: " + ", ".join(
            f"{g}={a/b:.0%}(n={b//len(small)})" for g, (a, b) in sorted(agg.items(), key=lambda x: str(x[0]))))
    gap = [i for i, t in per_task.items()
           if strong and all(t.get(k) == "OK" for k in strong) and all(t.get(k) != "OK" for k in small)]
    print(f"-- tasks all strong pass but all small fail: {len(gap)}/{len(per_task)}")
    return gap


if __name__ == "__main__":
    out_dir = sys.argv[1]
    domains = sys.argv[2:] or sorted(os.listdir(out_dir))
    os.makedirs(os.path.join(os.path.dirname(__file__), "out"), exist_ok=True)
    for d in domains:
        rows, per_task = analyze(out_dir, d)
        gap = report(d, rows, per_task)
        json.dump({"rows": rows, "per_task": {i: {k: v for k, v in t.items()} for i, t in per_task.items()},
                   "gap_tasks": gap},
                  open(os.path.join(os.path.dirname(__file__), "out", f"taxonomy_{d}.json"), "w"),
                  indent=1, ensure_ascii=False, default=list)

In [ ]:
%%writefile /kaggle/working/frameguard/analysis/alignment.py
"""Alignment tối ưu cho nhóm Precedence của DECLARE và phản hồi sinh từ alignment (dùng chung mọi domain).

Quyết định thiết kế (CLAUDE.md): Gate chỉ ép Precedence / Branched / Choice / Chain; không tự phán điều kiện dữ liệu.

Với mô hình chỉ gồm các template này, alignment tối ưu tại một lần kích hoạt X là:
  Act Y chưa thỏa  -> sửa(Y.sub) + [chèn Y]                 (model move, đệ quy theo Chain)
  and              -> nối các sửa của từng phần (bỏ trùng)
  or               -> nhánh có chi phí nhỏ nhất (1/bước; hòa: tránh internal_get_database, ít tham số phải hỏi)
Thứ tự các bước chèn là một thứ tự tô-pô hợp lệ (tiền điều kiện trước).

Chạy offline: .venv/bin/python analysis/alignment.py [domain ...]
  (1) Thống kê alignment tại lần chặn ĐẦU TIÊN trên trace thật của model nhỏ.
  (2) Độ khớp lựa chọn nhánh với trace thành công của model mạnh.
  (3) "Agent lý tưởng": chỉ làm theo phản hồi alignment (+ đọc kết quả kiểm tra) có pass oracle không → kiểm tra tính đủ.
"""
from __future__ import annotations
import collections, copy, json, os, sys
from dataclasses import dataclass

sys.path.insert(0, os.path.dirname(__file__))
from declare_rules import (SOPBENCH, Act, Op, load_rules, new_strict_system, domain_keys,  # noqa: E402
                           domain_assistant_keys)
import conformance as C  # noqa: E402
from failure_taxonomy import SMALL, STRONG, SUFFIX  # noqa: E402

DB_DUMP = "internal_get_database"


# ---------------------------------------------------------------- alignment
@dataclass
class Move:
    tool: str
    args: dict            # tham số suy được từ tương quan với X
    ask_user: list        # tham số không suy được -> phải hỏi người dùng

    def key(self):
        return (self.tool, tuple(sorted((k, C.norm(v)) for k, v in self.args.items())))


def _cost(moves):
    # Đọc toàn bộ DB tính 2 bước: ngữ cảnh rất dài, model nhỏ dễ lạc; ưu tiên tool kiểm tra cụ thể.
    n_dump = sum(m.tool == DB_DUMP for m in moves)
    return (len(moves) + n_dump, n_dump, sum(len(m.ask_user) for m in moves))


def state_tools(domain):
    """Tool tạo trạng thái (login, xác thực): chỉ tính là đã thỏa khi KẾT QUẢ THẬT trả về thành công."""
    cl = domain_assistant_keys[domain].constraint_links
    return {(v[0] if isinstance(v, tuple) else v) for v in cl.values()}


def record(rs, hist, name, args, res):
    """Cập nhật lịch sử theo ngữ nghĩa của Gate (chỉ dùng kết quả tool thật)."""
    hist.any[name].append(args)
    if name in state_tools(rs.domain) and not C.succeeded(res):
        return
    hist.conformant[name].append(args)
    C.add_implied(rs, name, args, hist)


def repair(f, binding, hist):
    """Chuỗi model move tối thiểu để công thức tiền điều kiện f được thỏa."""
    if f is None:
        return []
    if isinstance(f, Act):
        if hist.match(f, binding, hist.conformant):
            return []
        args = {k: binding[v] for k, v in f.params.items() if binding.get(v) is not None}
        ask = [k for k, v in f.params.items() if binding.get(v) is None]
        return _dedupe(repair(f.sub, binding, hist) + [Move(f.name, args, ask)])
    if f.kind == "and":
        return _dedupe([m for p in f.parts for m in repair(p, binding, hist)])
    return min((repair(p, binding, hist) for p in f.parts), key=_cost)


def _dedupe(moves):
    seen, out = set(), []
    for m in moves:
        if m.key() not in seen:
            seen.add(m.key())
            out.append(m)
    return out


def align_activation(rs, name, args, hist):
    """Alignment cho lời gọi X=name(args) trên lịch sử hiện tại. Rỗng = không chặn."""
    f = rs.precondition(name)
    if not isinstance(f, Act):
        return []
    binding = {v: args.get(k) for k, v in f.params.items()}
    return repair(f.sub, binding, hist)


# ---------------------------------------------------------------- phản hồi
def describe(domain, tool):
    d = domain_assistant_keys[domain].action_descriptions.get(tool, "")
    return d.split(".")[0] if d else ""


def render_alignment(domain, blocked, moves, observed=None):
    """Phản hồi kiểu `alignment`: chuỗi bước tối thiểu, kèm tham số đã biết và tham số cần hỏi người dùng."""
    lines = [f"BLOCKED: `{blocked}` cannot run yet. Minimal steps to make it allowed, in order:"]
    for i, m in enumerate(moves, 1):
        a = ", ".join(f"{k}={json.dumps(v)}" for k, v in m.args.items())
        ask = f"; ask the user for: {', '.join(m.ask_user)}" if m.ask_user else ""
        lines.append(f"  {i}. call `{m.tool}`({a}){ask}")
    lines.append(f"Then call `{blocked}` again only if every check above succeeds; if a check fails, tell the user the request cannot be completed.")
    if observed:
        lines.append("Results you already have: " + "; ".join(f"{t} -> {r}" for t, r in observed))
    return "\n".join(lines)


def render_template(domain, blocked, moves):
    """Phản hồi kiểu `template` (một câu cho ràng buộc đầu tiên chưa thỏa) — để so sánh độ dài/nội dung."""
    m = moves[0]
    return f"BLOCKED: Precedence violated — you must call `{m.tool}` before `{blocked}`."


# ---------------------------------------------------------------- (1) thống kê trên trace thật
def first_block(rs, interaction):
    hist = C.History()
    for c in C.extract_calls(interaction):
        if c["name"] not in rs.action_params:
            continue
        moves = align_activation(rs, c["name"], c["args"], hist)
        if moves:
            return c, moves
        record(rs, hist, c["name"], c["args"], c["result"])
    return None, None


# ---------------------------------------------------------------- (3) agent lý tưởng
def goal_args(rs):
    node = rs.goal_graph["nodes"][0][1]
    uk = rs.task["user_known"]
    return {k: uk[v] for k, v in node.items() if v in uk}


def ideal_follower(rs, max_rounds=10):
    """Agent chỉ làm theo alignment; tham số cần hỏi lấy từ user_known (người dùng trả lời đúng).
    Sau khi đủ tiền điều kiện, quyết định gọi goal theo kết quả kiểm tra thật (mô phỏng model đọc kết quả đúng)."""
    from env.evaluator import evaluator_function_directed_graph
    env = domain_keys[rs.domain](copy.deepcopy(rs.task["initial_database"]),
                                 domain_assistant_keys[rs.domain].action_innate_dependencies,
                                 rs.task["constraint_parameters"])
    strict = new_strict_system(rs)
    uk = rs.task["user_known"]
    hist, calls, n_blocks = C.History(), [], 0
    gargs = goal_args(rs)

    def do(tool, args):
        try:
            res = copy.deepcopy(getattr(env, tool)(**args))
        except Exception as e:
            res = f"Error: {e}"
        try:
            getattr(strict, tool)(**args)
        except Exception:
            pass
        calls.append({"tool_name": tool, "arguments": args, "content": res})
        record(rs, hist, tool, args, res)
        return res

    refused = False
    for _ in range(max_rounds):
        moves = align_activation(rs, rs.goal, gargs, hist)
        if not moves:
            break
        n_blocks += 1
        for m in moves:
            a = dict(m.args)
            for k in m.ask_user:
                # tham số hỏi người dùng: khớp theo tên tham số trong user_known
                if k in uk:
                    a[k] = uk[k]
            res = do(m.tool, a)
            if m.tool in state_tools(rs.domain) and not C.succeeded(res):
                refused = True             # bước kiểm tra thất bại -> phản hồi bảo từ chối
                break
        if refused:
            break
    else:
        return {"ok": False, "reason": "no_convergence", "blocks": n_blocks}
    try:
        allowed = (not refused) and strict.evaluation_get_dependency_evaluator().process(method_str=rs.goal, **gargs)
    except Exception:
        allowed = False
    if allowed:
        do(rs.goal, gargs)
    ev = evaluator_function_directed_graph(rs.domain, rs.task, [], calls,
                                           {"final_database": env.evaluation_get_database()}, "full")
    reason = "pass" if ev["success"] else ",".join(k for k in ("no_tool_call_error", "constraint_not_violated", "database_match",
                                                             "action_called_correctly", "dirgraph_satisfied") if not ev[k])
    return {"ok": ev["success"], "reason": reason, "blocks": n_blocks, "n_moves": len(calls) - int(allowed)}


# ---------------------------------------------------------------- chạy
def run(domain):
    bad = C.unsolvable(domain)
    data = None
    # (1) lần chặn đầu tiên ở model nhỏ
    stat = collections.defaultdict(collections.Counter)
    example = None
    for key in SMALL:
        p = os.path.join(SOPBENCH, "output", domain, f"ast_{key}{SUFFIX}")
        if not os.path.exists(p):
            continue
        d = json.load(open(p))
        data = data or d
        for i, t in enumerate(d):
            if i in bad or t["evaluations"][0]["success"]:
                continue
            rs = load_rules(domain, t["task"])
            c, moves = first_block(rs, t["interactions"][0]["interaction"])
            if not moves:
                stat[key]["no_block"] += 1
                continue
            stat[key]["blocked"] += 1
            stat[key][f"len={min(len(moves), 4)}{'+' if len(moves) >= 4 else ''}"] += 1
            stat[key]["needs_user_info"] += any(m.ask_user for m in moves)
            stat[key]["uses_db_dump"] += any(m.tool == DB_DUMP for m in moves)
            if example is None and len(moves) >= 2:
                example = (c["name"], render_alignment(domain, c["name"], moves), render_template(domain, c["name"], moves))
    # (2) lựa chọn nhánh so với trace thành công của model mạnh
    match = collections.Counter()
    for key in STRONG:
        p = os.path.join(SOPBENCH, "output", domain, f"ast_{key}{SUFFIX}")
        if not os.path.exists(p):
            continue
        for i, t in enumerate(json.load(open(p))):
            if i in bad or not t["evaluations"][0]["success"] or not t["task"]["action_should_succeed"]:
                continue
            rs = load_rules(domain, t["task"])
            moves = align_activation(rs, rs.goal, goal_args(rs), C.History())
            called = {c["name"] for c in C.extract_calls(t["interactions"][0]["interaction"])}
            for m in moves:
                match["suggested_used" if m.tool in called else "suggested_not_used"] += 1
    # (3) agent lý tưởng trên mọi task sạch
    ideal = collections.Counter()
    fails = []
    for i, t in enumerate(data or []):
        if i in bad:
            continue
        rs = load_rules(domain, t["task"])
        r = ideal_follower(rs)
        ideal[r["reason"]] += 1
        ideal[f"blocks={r['blocks']}"] += 1
        if not r["ok"] and len(fails) < 5:
            fails.append((i, rs.goal, rs.should_succeed, r["reason"]))
    return stat, match, ideal, fails, example


if __name__ == "__main__":
    domains = sys.argv[1:] or sorted(os.listdir(os.path.join(SOPBENCH, "output")))
    out = {}
    for dom in domains:
        stat, match, ideal, fails, example = run(dom)
        n = sum(v for k, v in ideal.items() if not k.startswith("blocks="))
        print(f"\n===== {dom} =====")
        print(f"  (3) agent lý tưởng theo alignment: pass {ideal['pass']}/{n} = {ideal['pass'] / max(1, n):.1%}   "
              + ", ".join(f"{k}:{v}" for k, v in sorted(ideal.items()) if k != "pass"))
        for f in fails:
            print("      fail:", f)
        tot = match["suggested_used"] + match["suggested_not_used"]
        print(f"  (2) bước alignment đề xuất có trong trace thành công của model mạnh: {match['suggested_used']}/{tot} = {match['suggested_used'] / max(1, tot):.1%}")
        print("  (1) lần chặn đầu tiên trong trace thất bại của model nhỏ:")
        for k, s in stat.items():
            b = s["blocked"]
            print(f"      {k:32s} bị chặn {b:3d} | độ dài sửa " + " ".join(f"{L}:{s[L]}" for L in ("len=1", "len=2", "len=3", "len=4+"))
                  + f" | cần hỏi user {s['needs_user_info'] / max(1, b):.0%} | dùng DB dump {s['uses_db_dump'] / max(1, b):.0%}")
        if example:
            print(f"  Ví dụ phản hồi khi chặn `{example[0]}`:\n    [template] {example[2]}\n    [alignment]\n      " + example[1].replace("\n", "\n      "))
        out[dom] = {"ideal": dict(ideal), "match": dict(match), "first_block": {k: dict(v) for k, v in stat.items()}}
    os.makedirs(C.OUT, exist_ok=True)
    json.dump(out, open(os.path.join(C.OUT, "alignment_summary.json"), "w"), indent=1, ensure_ascii=False)

In [ ]:
%%writefile /kaggle/working/frameguard/analysis/oracle_fix.py
"""Sửa lỗi oracle SOPBench: harness mới (swarm/core.py) bỏ phần tử đầu của kết quả tuple (True, value) trước khi ghi log,
nên evaluator không bao giờ thấy goal 'thành công' → task should_succeed có goal trả tuple không thể pass.
Sửa: goal được gọi, kết quả không phải bool False và không phải chuỗi lỗi → coi là thành công. Các tiêu chí khác giữ nguyên."""
import json, os, re, sys, collections
sys.path.insert(0, os.path.dirname(__file__))
import conformance as C
from failure_taxonomy import SMALL, STRONG, SUFFIX
from declare_rules import SOPBENCH

ERR = re.compile(r"^(Error|[A-Z][A-Za-z]*Error|[A-Z][A-Za-z]*Exception)\b")

def goal_succeeded_fixed(task, interaction):
    for c in C.extract_calls(interaction):
        if c["name"] != task["user_goal"]:
            continue
        r = c["result"]
        if isinstance(r, bool):
            ok = r
        elif isinstance(r, (tuple, list)) and r and isinstance(r[0], bool):
            ok = r[0]
        elif isinstance(r, str) and (ERR.match(r) or r.strip() == "False"):
            ok = False
        else:
            ok = True          # giá trị trả về của tuple (True, value) đã bị bỏ cờ
        if ok:
            return True
    return False

def fixed_success(t):
    ev, task = t["evaluations"][0], t["task"]
    asc = goal_succeeded_fixed(task, t["interactions"][0]["interaction"])
    return (ev["no_tool_call_error"] and ev["constraint_not_violated"] and ev["database_match"]
            and ev["dirgraph_satisfied"] and asc == task["action_should_succeed"])

if __name__ == "__main__":
    rows = collections.defaultdict(lambda: [0, 0, 0])
    for dom in sorted(os.listdir(os.path.join(SOPBENCH, "output"))):
        bad = C.unsolvable(dom)
        for key in SMALL + STRONG:
            p = os.path.join(SOPBENCH, "output", dom, f"ast_{key}{SUFFIX}")
            if not os.path.exists(p): continue
            d = json.load(open(p))
            if not all(t.get("evaluations") for t in d): continue
            for i, t in enumerate(d):
                if i in bad: continue
                rows[key][0] += 1; rows[key][1] += t["evaluations"][0]["success"]; rows[key][2] += fixed_success(t)
    print(f"{'model':34s} {'n':>5s} {'pass gốc':>9s} {'pass sửa':>9s} {'chênh':>7s}")
    for k, (n, a, b) in rows.items():
        print(f"{k:34s} {n:5d} {a/n:9.1%} {b/n:9.1%} {(b-a)/n:+7.1%}")

In [ ]:
%%writefile /kaggle/working/frameguard/frameguard/__init__.py
"""FrameGuard: Gate tất định + phản hồi theo DECLARE cho LLM agent cỡ nhỏ."""

In [ ]:
%%writefile /kaggle/working/frameguard/frameguard/feedback.py
"""Sinh tin nhắn phản hồi khi Gate chặn. Mọi câu chữ sinh từ template, không viết tay theo task.

Các kiểu: bare / reason / next_action / template / alignment (xem CLAUDE.md, RQ2).
Tiếng Anh vì prompt và tool của SOPBench là tiếng Anh.
"""
import json

MODES = ("none", "bare", "reason", "next_action", "template", "alignment")


def _args(a):
    return ", ".join(f"{k}={json.dumps(v)}" for k, v in a.items())


def render(mode, blocked, violations, moves, failed_state):
    """
    blocked       : tên tool bị chặn
    violations    : [(template, (sources...), kind)] — ràng buộc DECLARE chưa thỏa
    moves         : chuỗi sửa tối thiểu (alignment.Move), theo thứ tự
    failed_state  : tool tạo trạng thái đã gọi nhưng thất bại (vd login_user trả False)
    """
    if mode == "bare":
        return f"Error: the call to `{blocked}` was blocked."
    if mode == "reason":
        return (f"Error: the call to `{blocked}` was blocked because it violates the standard operating procedure: "
                f"a required prerequisite has not been satisfied.")
    if mode == "next_action":
        if failed_state:
            return (f"Error: the call to `{blocked}` was blocked. Next action: tell the user the request cannot be "
                    f"completed, because `{failed_state[0]}` did not succeed.")
        m = moves[0]
        return f"Error: the call to `{blocked}` was blocked. Next action: call `{m.tool}`."
    if mode == "template":
        lines = [f"Error: the call to `{blocked}` was blocked by the procedure."]
        for tpl, src, kind in violations:
            fs = [s for s in src if s in failed_state]
            if fs:
                lines.append(f"- `{fs[0]}` was called but did not succeed, so the requirement for `{blocked}` is not met. "
                             f"Do not retry `{blocked}`; tell the user the request cannot be completed.")
            elif tpl == "Precedence":
                lines.append(f"- Precedence: you must successfully call `{src[0]}` before `{blocked}`.")
            else:
                alts = " or ".join(f"`{s}`" for s in src)
                lines.append(f"- You must successfully call one of {alts} before `{blocked}`.")
        lines.append("Do not repeat the blocked call unchanged.")
        return "\n".join(lines)
    if mode == "alignment":
        if failed_state:
            return (f"Error: the call to `{blocked}` was blocked. `{failed_state[0]}` was called but did not succeed, "
                    f"so `{blocked}` is not allowed. Do not retry it; tell the user the request cannot be completed.")
        lines = [f"Error: the call to `{blocked}` was blocked. Minimal steps to make it allowed, in order:"]
        for i, m in enumerate(moves, 1):
            ask = (f"; use the user's {', '.join(m.ask_user)} (ask the user only if you do not have it)"
                   if m.ask_user else "")
            lines.append(f"  {i}. call `{m.tool}`({_args(m.args)}){ask}")
        lines.append(f"Then call `{blocked}` again only if every check above succeeds; "
                     f"if a check fails, tell the user the request cannot be completed.")
        return "\n".join(lines)
    raise ValueError(mode)

In [ ]:
%%writefile /kaggle/working/frameguard/frameguard/gate.py
"""Gate tất định (cách 1): chỉ ép nhóm Precedence, trạng thái chỉ cập nhật từ kết quả tool thật.

Dùng phần lõi đã kiểm định ở giai đoạn 1 (analysis/alignment.py, analysis/conformance.py).
Sẽ tách hẳn khỏi analysis/ khi tái cấu trúc frameguard/ + adapters/.
"""
import collections, os, sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "analysis"))
import conformance as C  # noqa: E402
from alignment import align_activation, record, state_tools  # noqa: E402
from declare_rules import Act  # noqa: E402
from frameguard.feedback import render  # noqa: E402


class Gate:
    def __init__(self, rules, mode):
        self.rs, self.mode = rules, mode
        self.hist = C.History()
        self.blocked_counts = collections.Counter()
        self.log = []                       # một dòng cho mỗi tool call đi qua Gate

    def check(self, name, args):
        """Trả về tin nhắn phản hồi nếu chặn, None nếu cho chạy."""
        if self.mode == "none" or name not in self.rs.action_params or not isinstance(args, dict):
            return None
        moves = align_activation(self.rs, name, args, self.hist)
        if not moves:
            return None
        f = self.rs.precondition(name)
        binding = {v: args.get(k) for k, v in f.params.items()} if isinstance(f, Act) else {}
        viol = C.unsatisfied(f.sub if isinstance(f, Act) else None, binding, self.hist)
        st = state_tools(self.rs.domain)
        failed = [m.tool for m in moves if m.tool in st and m.tool in self.hist.any]
        key = (name, C.norm(args))
        self.blocked_counts[key] += 1
        msg = render(self.mode, name, viol, moves, failed)
        self.log.append({"tool": name, "args": args, "decision": "blocked",
                         "violations": [list(v[:1]) + [list(v[1])] + [v[2]] for v in viol],
                         "repair": [m.tool for m in moves], "failed_state": failed,
                         "repeat_index": self.blocked_counts[key], "message": msg})
        return msg

    def observe(self, name, args, content):
        """Cập nhật trạng thái từ kết quả tool thật (kể cả khi mode=none, để có log so sánh)."""
        if name in self.rs.action_params and isinstance(args, dict):
            record(self.rs, self.hist, name, args, C.try_eval(content))
        self.log.append({"tool": name, "args": args, "decision": "allowed", "result": str(content)[:300]})

    def summary(self):
        b = [e for e in self.log if e["decision"] == "blocked"]
        return {"n_blocked": len(b), "n_repeat_blocked": sum(e["repeat_index"] > 1 for e in b),
                "templates": collections.Counter(v[0] for e in b for v in e["violations"])}

In [ ]:
%%writefile /kaggle/working/frameguard/adapters/sopbench/runner.py
"""Chạy SOPBench với FrameGuard Gate (adapter SOPBench).

Giữ nguyên cấu hình công bố của SOPBench (fc, tool_list full, env_mode prompt, constraint full/structured,
không user simulator, temperature 0, top_p 0.01, max_tokens 1024, max_turns 20, max_actions 10).
Không sửa mã SOPBench: Gate gắn bằng cách bọc Swarm.handle_tool_calls; kiểu dữ liệu vLLM đổi bằng cách bọc lệnh khởi động.

Ví dụ:
  python adapters/sopbench/runner.py --model qwen3.5-2b --domain bank --modes none,bare,alignment --task_set smoke
  python adapters/sopbench/runner.py --fake_replay qwen3.5-2b-mode_fc --domain bank --modes none,alignment --task_set smoke
"""
import argparse, copy, json, os, sys, time, traceback

ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
SOPBENCH = os.environ.setdefault("SOPBENCH", os.path.join(ROOT, "external", "SOPBench"))
sys.path[:0] = [ROOT, os.path.join(ROOT, "analysis"), SOPBENCH]
os.chdir(SOPBENCH)  # SOPBench dùng đường dẫn tương đối

import swarm.llm_handler as lh  # noqa: E402
import swarm.core as core  # noqa: E402
from swarm.types import Agent, Response  # noqa: E402
from swarm.util import function_to_json, construct_chatcompletion  # noqa: E402
import run_simulation as rsim  # noqa: E402
from env.task import task_default_dep_full  # noqa: E402
from env.evaluator import evaluator_function_directed_graph  # noqa: E402
from openai.types.chat import ChatCompletionMessageToolCall  # noqa: E402
from openai.types.chat.chat_completion_message_tool_call import Function  # noqa: E402

from declare_rules import load_rules  # noqa: E402
import conformance as C  # noqa: E402
from oracle_fix import goal_succeeded_fixed  # noqa: E402
from frameguard.gate import Gate  # noqa: E402

# ------------------------------------------------------------------ vLLM: đổi dtype / max-model-len khi cần (T4 không có bf16)
_Popen = lh.subprocess.Popen


def _popen(cmd, *a, **k):
    if isinstance(cmd, list) and cmd[:2] == ["vllm", "serve"]:
        cmd = list(cmd)
        if os.getenv("VLLM_DTYPE") and "--dtype" in cmd:
            cmd[cmd.index("--dtype") + 1] = os.environ["VLLM_DTYPE"]
        if os.getenv("VLLM_MAX_MODEL_LEN") and "--max-model-len" in cmd:
            cmd[cmd.index("--max-model-len") + 1] = os.environ["VLLM_MAX_MODEL_LEN"]
        print("[frameguard] vllm cmd:", " ".join(cmd), flush=True)
    return _Popen(cmd, *a, **k)


lh.subprocess.Popen = _popen

# ------------------------------------------------------------------ Gate gắn vào Swarm.handle_tool_calls
CURRENT_GATE = None
_orig_handle = core.Swarm.handle_tool_calls


def _gated_handle(self, tool_calls, functions, context_variables, debug):
    G = CURRENT_GATE
    msgs = []
    for tc in tool_calls:
        name = tc.function.name
        try:
            args = json.loads(tc.function.arguments)
        except Exception:
            args = None
        msg = G.check(name, args) if G else None
        if msg is not None:
            msgs.append({"role": "tool", "tool_call_id": tc.id, "tool_name": name, "content": msg})
            continue
        resp = _orig_handle(self, [tc], functions, context_variables, debug)
        for m in resp.messages:
            if G:
                G.observe(name, args, m.get("content"))
        msgs.extend(resp.messages)
    return Response(messages=msgs, agent=None, context_variables={})


core.Swarm.handle_tool_calls = _gated_handle


# ------------------------------------------------------------------ model giả: phát lại trajectory công bố (kiểm tra ống dẫn, không cần GPU)
class ReplayHandler(lh.OpenAIHandler):  # kế thừa để qua kiểm tra kiểu của Agent; không gọi __init__ gốc
    def __init__(self, domain, model_key):
        p = os.path.join(SOPBENCH, "output", domain, f"ast_{model_key}-dep_full-fmt_structured-tool_full-shuffle_False.json")
        self.data = json.load(open(p))
        self.model_name = self.model_name_huggingface = f"replay:{model_key}"
        self.script = []

    def start(self, idx):
        self.script = [m for m in self.data[idx]["interactions"][0]["interaction"]
                       if m.get("sender", "").endswith("assistant")]

    def inference(self, create_params, debug, mode="chat", tool_call_mode="fc"):
        if not self.script:
            tc = [ChatCompletionMessageToolCall(id="end", type="function", function=Function(name="exit_conversation", arguments="{}"))]
            return {"completion": construct_chatcompletion("assistant", "", tc)}
        m = self.script.pop(0)
        tcs = [ChatCompletionMessageToolCall(id=t["id"], type="function",
                                             function=Function(name=t["function"]["name"], arguments=t["function"]["arguments"]))
               for t in (m.get("tool_calls") or [])] or None
        return {"completion": construct_chatcompletion("assistant", m.get("content") or "", tcs)}

    def kill_process(self):
        pass

    def __del__(self):
        pass


# ------------------------------------------------------------------ chạy
def sop_args(a):
    argv = ["--domain", a.domain, "--assistant_model", a.model or "replay", "--tool_call_mode", "fc",
            "--tool_list", "full", "--env_mode", "prompt", "--default_constraint_option", "full",
            "--constraint_descr_format", "structured", "--assistant_temperature", "0.0", "--assistant_top_p", "0.01",
            "--assistant_max_tokens", str(a.max_tokens), "--max_num_turns", "20", "--max_num_actions", str(a.max_actions)]
    old = sys.argv
    sys.argv = ["run_simulation.py"] + argv
    try:
        return rsim.parse_args()
    finally:
        sys.argv = old


def load_tasks(domain):
    tasks_dict = json.load(open(os.path.join(SOPBENCH, "data", f"{domain}_tasks.json")))
    tasks = []
    for k, arr in tasks_dict.items():
        for t in copy.deepcopy(arr):
            t["user_goal"] = k
            tasks.append(t)
    return tasks


def select(a):
    sets = json.load(open(os.path.join(ROOT, "adapters", "sopbench", "task_sets.json")))
    if a.task_set == "smoke":
        s = sets[f"smoke_{a.domain}_qwen3.5-2b"]
        return s["gate_addressable_failures"] + s["published_passes"]
    if a.task_set == "clean":
        return sets[a.domain]["clean"]
    return [int(x) for x in a.task_set.split(",")]


def evaluate(domain, task, interaction, database):
    func_calls = [{"tool_name": c["name"], "arguments": c["args"], "content": c["result"]}
                  for c in C.extract_calls(interaction)]
    ev = evaluator_function_directed_graph(domain, task, interaction, func_calls, {"final_database": database}, "full")
    asc_fixed = goal_succeeded_fixed(task, interaction)
    ev["success_fixed"] = (ev["no_tool_call_error"] and ev["constraint_not_violated"] and ev["database_match"]
                           and ev["dirgraph_satisfied"] and asc_fixed == task["action_should_succeed"])
    return ev


def main():
    global CURRENT_GATE
    p = argparse.ArgumentParser()
    p.add_argument("--domain", default="bank")
    p.add_argument("--model", default=None, help="tên model SOPBench, vd qwen3.5-2b, gemma-4-e2b-it")
    p.add_argument("--fake_replay", default=None, help="vd qwen3.5-2b-mode_fc: phát lại trajectory công bố thay cho model")
    p.add_argument("--modes", default="none,bare,alignment")
    p.add_argument("--task_set", default="smoke", help="smoke | clean | danh sách id cách nhau dấu phẩy")
    p.add_argument("--max_tokens", type=int, default=1024)
    p.add_argument("--max_actions", type=int, default=10)
    p.add_argument("--num_gpus", type=int, default=1)
    p.add_argument("--out", default=os.path.join(ROOT, "runs"))
    a = p.parse_args()

    args = sop_args(a)
    tasks = load_tasks(a.domain)
    ids = select(a)
    dep_innate_full, default_dep_full, default_dep_full_descr = task_default_dep_full(
        a.domain, "full", "structured", dependency_verb_dep_orig=True)

    if a.fake_replay:
        handler = ReplayHandler(a.domain, a.fake_replay)
        run_name = f"replay_{a.fake_replay}"
    else:
        handler = lh.OpenAIHandler(model_name=a.model, num_gpus=a.num_gpus, gpu_memory_utilization=0.92, tool_calling=True)
        run_name = a.model
    assistant = Agent(name="domain assistant", client=handler, temperature=0.0, top_p=0.01,
                      max_tokens=a.max_tokens, tool_call_mode="fc", instructions="", functions=[])
    user = Agent(name="user", client=None, default_response="", response_repeat=True)

    summary = []
    try:
        for mode in a.modes.split(","):
            out_dir = os.path.join(a.out, a.domain, run_name, mode)
            os.makedirs(out_dir, exist_ok=True)
            for i in ids:
                f = os.path.join(out_dir, f"task_{i:04d}.json")
                if os.path.exists(f):           # chạy tiếp được khi phiên bị ngắt
                    summary.append(json.load(open(f))["summary"])
                    continue
                task = copy.deepcopy(tasks[i])
                CURRENT_GATE = Gate(load_rules(a.domain, task), mode)
                if a.fake_replay:
                    handler.start(i)
                t0 = time.time()
                try:
                    res = rsim.run_task_simulation(args=args, task=task, dep_innate_full=dep_innate_full,
                                                   default_dep_full=default_dep_full,
                                                   default_dep_full_descr=default_dep_full_descr,
                                                   assistant_agent=assistant, user_agent=user)
                except Exception as e:
                    traceback.print_exc()
                    res = {"interaction": [], "database": task["initial_database"], "error": repr(e)}
                inter = res["interaction"]
                for m in inter:
                    m.pop("role", None)
                ev = evaluate(a.domain, task, inter, res["database"])
                gs = CURRENT_GATE.summary()
                s = {"task": i, "goal": task["user_goal"], "should_succeed": task["action_should_succeed"], "mode": mode,
                     "success": ev["success"], "success_fixed": ev["success_fixed"], "n_blocked": gs["n_blocked"],
                     "n_repeat_blocked": gs["n_repeat_blocked"], "n_tool_calls": ev["num_function_calls"],
                     "seconds": round(time.time() - t0, 1)}
                json.dump({"config": {"model": run_name, "domain": a.domain, "mode": mode, "vllm_dtype": os.getenv("VLLM_DTYPE", "bfloat16"),
                                      "thinking": {"qwen": os.getenv("QWEN35_THINKING"), "gemma": os.getenv("GEMMA4_THINKING")},
                                      "max_tokens": a.max_tokens, "max_actions": a.max_actions},
                           "task": task, "interaction": inter, "database": res["database"], "evaluation": ev,
                           "gate_log": CURRENT_GATE.log, "summary": s},
                          open(f, "w"), indent=1, ensure_ascii=False, default=str)
                summary.append(s)
                print(f"[frameguard] {mode:11s} task {i:3d} {task['user_goal']:28s} success={ev['success']!s:5s} "
                      f"fixed={ev['success_fixed']!s:5s} blocked={gs['n_blocked']} {s['seconds']}s", flush=True)
    finally:
        if hasattr(handler, "kill_process"):
            handler.kill_process()
    report(summary)


def report(summary):
    import collections
    by = collections.defaultdict(list)
    for s in summary:
        by[s["mode"]].append(s)
    print("\nmode         n   pass  pass(sửa)  bị chặn≥1  phục hồi*  lặp-chặn  giây/task")
    for mode, rows in by.items():
        n = len(rows)
        blocked = [r for r in rows if r["n_blocked"] > 0]
        rec = sum(r["success_fixed"] for r in blocked)
        print(f"{mode:11s} {n:3d} {sum(r['success'] for r in rows):5d} {sum(r['success_fixed'] for r in rows):9d} "
              f"{len(blocked):10d} {f'{rec}/{len(blocked)}':>10s} {sum(r['n_repeat_blocked'] for r in rows):9d} "
              f"{sum(r['seconds'] for r in rows) / max(1, n):10.1f}")
    print("* phục hồi = số task pass (oracle đã sửa) trong các task bị chặn ít nhất một lần")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/frameguard/adapters/sopbench/task_sets.json
{
 "bank": {
  "n_total": 134,
  "excluded": [
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   39,
   44,
   56,
   61,
   62,
   63,
   64,
   65,
   66,
   76,
   89,
   120
  ],
  "clean": [
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   40,
   41,
   42,
   43,
   45,
   46,
   47,
   48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   57,
   58,
   59,
   60,
   67,
   68,
   69,
   70,
   71,
   72,
   73,
   74,
   75,
   77,
   78,
   79,
   80,
   81,
   82,
   83,
   84,
   85,
   86,
   87,
   88,
   90,
   91,
   92,
   93,
   94,
   95,
   96,
   97,
   98,
   99,
   100,
   101,
   102,
   103,
   104,
   105,
   106,
   107,
   108,
   109,
   110,
   111,
   112,
   113,
   114,
   115,
   116,
   117,
   118,
   119,
   121,
   122,
   123,
   124,
   125,
   126,
   127,
   128,
   129,
   130,
   131,
   132,
   133
  ]
 },
 "dmv": {
  "n_total": 97,
  "excluded": [
   84
  ],
  "clean": [
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   44,
   45,
   46,
   47,
   48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65,
   66,
   67,
   68,
   69,
   70,
   71,
   72,
   73,
   74,
   75,
   76,
   77,
   78,
   79,
   80,
   81,
   82,
   83,
   85,
   86,
   87,
   88,
   89,
   90,
   91,
   92,
   93,
   94,
   95,
   96
  ]
 },
 "healthcare": {
  "n_total": 124,
  "excluded": [],
  "clean": [
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   44,
   45,
   46,
   47,
   48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65,
   66,
   67,
   68,
   69,
   70,
   71,
   72,
   73,
   74,
   75,
   76,
   77,
   78,
   79,
   80,
   81,
   82,
   83,
   84,
   85,
   86,
   87,
   88,
   89,
   90,
   91,
   92,
   93,
   94,
   95,
   96,
   97,
   98,
   99,
   100,
   101,
   102,
   103,
   104,
   105,
   106,
   107,
   108,
   109,
   110,
   111,
   112,
   113,
   114,
   115,
   116,
   117,
   118,
   119,
   120,
   121,
   122,
   123
  ]
 },
 "hotel": {
  "n_total": 195,
  "excluded": [
   46,
   70,
   80,
   100,
   110,
   122,
   133,
   146,
   154,
   159,
   163,
   165,
   167,
   170,
   172,
   174,
   176,
   194
  ],
  "clean": [
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   44,
   45,
   47,
   48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65,
   66,
   67,
   68,
   69,
   71,
   72,
   73,
   74,
   75,
   76,
   77,
   78,
   79,
   81,
   82,
   83,
   84,
   85,
   86,
   87,
   88,
   89,
   90,
   91,
   92,
   93,
   94,
   95,
   96,
   97,
   98,
   99,
   101,
   102,
   103,
   104,
   105,
   106,
   107,
   108,
   109,
   111,
   112,
   113,
   114,
   115,
   116,
   117,
   118,
   119,
   120,
   121,
   123,
   124,
   125,
   126,
   127,
   128,
   129,
   130,
   131,
   132,
   134,
   135,
   136,
   137,
   138,
   139,
   140,
   141,
   142,
   143,
   144,
   145,
   147,
   148,
   149,
   150,
   151,
   152,
   153,
   155,
   156,
   157,
   158,
   160,
   161,
   162,
   164,
   166,
   168,
   169,
   171,
   173,
   175,
   177,
   178,
   179,
   180,
   181,
   182,
   183,
   184,
   185,
   186,
   187,
   188,
   189,
   190,
   191,
   192,
   193
  ]
 },
 "library": {
  "n_total": 66,
  "excluded": [
   0,
   36,
   44,
   47,
   48,
   51
  ],
  "clean": [
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   45,
   46,
   49,
   50,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65
  ]
 },
 "online_market": {
  "n_total": 172,
  "excluded": [
   7
  ],
  "clean": [
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   44,
   45,
   46,
   47,
   48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65,
   66,
   67,
   68,
   69,
   70,
   71,
   72,
   73,
   74,
   75,
   76,
   77,
   78,
   79,
   80,
   81,
   82,
   83,
   84,
   85,
   86,
   87,
   88,
   89,
   90,
   91,
   92,
   93,
   94,
   95,
   96,
   97,
   98,
   99,
   100,
   101,
   102,
   103,
   104,
   105,
   106,
   107,
   108,
   109,
   110,
   111,
   112,
   113,
   114,
   115,
   116,
   117,
   118,
   119,
   120,
   121,
   122,
   123,
   124,
   125,
   126,
   127,
   128,
   129,
   130,
   131,
   132,
   133,
   134,
   135,
   136,
   137,
   138,
   139,
   140,
   141,
   142,
   143,
   144,
   145,
   146,
   147,
   148,
   149,
   150,
   151,
   152,
   153,
   154,
   155,
   156,
   157,
   158,
   159,
   160,
   161,
   162,
   163,
   164,
   165,
   166,
   167,
   168,
   169,
   170,
   171
  ]
 },
 "university": {
  "n_total": 42,
  "excluded": [],
  "clean": [
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41
  ]
 },
 "smoke_bank_qwen3.5-2b": {
  "gate_addressable_failures": [
   8,
   67,
   81,
   82,
   83,
   91,
   92,
   121
  ],
  "published_passes": [
   71,
   78,
   113,
   124
  ],
  "note": "T\u1eadp l\u00e0m gi\u00e0u \u0111\u1ec3 th\u1eed nhanh: KH\u00d4NG d\u00f9ng \u0111\u1ec3 \u01b0\u1edbc l\u01b0\u1ee3ng pass rate."
 }
}

## Bước 1 — kiểm tra 1 task (khởi động vLLM mất vài phút)
Nếu lỗi ở đây: xem log `[frameguard] vllm cmd` và thông báo của vLLM (thiếu bộ nhớ → bật `VLLM_MAX_MODEL_LEN`).

In [ ]:
!cd /kaggle/working/frameguard && python adapters/sopbench/runner.py --model {MODEL} --domain {DOMAIN} --modes none --task_set 8 2>&1 | grep -E "frameguard|Error|error|mode |^none" | tail -20

## Bước 2 — thử nhanh: 12 task × các chế độ

In [ ]:
!cd /kaggle/working/frameguard && python adapters/sopbench/runner.py --model {MODEL} --domain {DOMAIN} --modes {MODES} --task_set {TASK_SET} 2>&1 | grep -E "^\[frameguard\] (none|bare|reason|next|template|align)|^mode|^none|^bare|^reason|^next|^template|^alignment|phục hồi"

## Bước 3 — đọc kết quả

In [ ]:
import json, glob, collections
PUBLISHED = {8:False,67:False,81:False,82:False,83:False,91:False,92:False,121:False,71:True,78:True,113:True,124:True}  # qwen3.5-2b công bố
rows = [json.load(open(f))["summary"] for f in glob.glob(f"{WORK}/runs/{DOMAIN}/{MODEL}/*/task_*.json")]
by = collections.defaultdict(dict)
for r in rows: by[r["mode"]][r["task"]] = r
print("Task  goal                     công bố | " + " | ".join(f"{m:>9s}" for m in by))
for t, pub in PUBLISHED.items():
    goal = next((by[m][t]["goal"] for m in by if t in by[m]), "?")
    cells = []
    for m in by:
        r = by[m].get(t)
        cells.append("    -    " if r is None else f"{'PASS' if r['success_fixed'] else 'fail'} b={r['n_blocked']:<2d}")
    print(f"{t:4d}  {goal:24s} {'PASS' if pub else 'fail':>6s} | " + " | ".join(f"{c:>9s}" for c in cells))
if "none" in by:
    agree = sum(by["none"][t]["success_fixed"] == p for t, p in PUBLISHED.items() if t in by["none"])
    print(f"\nE0 — chế độ none khớp kết quả công bố: {agree}/{len(by['none'])}")
for m in by:
    fails = [t for t, p in PUBLISHED.items() if not p and t in by[m]]
    passes = [t for t, p in PUBLISHED.items() if p and t in by[m]]
    print(f"{m:10s}: cứu được {sum(by[m][t]['success_fixed'] for t in fails)}/{len(fails)} task từng thất bại; "
          f"giữ được {sum(by[m][t]['success_fixed'] for t in passes)}/{len(passes)} task từng pass; "
          f"trung bình {sum(r['seconds'] for r in by[m].values())/max(1,len(by[m])):.0f} giây/task")

In [ ]:
!cd /kaggle/working && zip -qr frameguard_runs.zip frameguard/runs && ls -lh frameguard_runs.zip

## Cách đọc
| Câu hỏi | Tín hiệu tích cực | Nếu không đạt |
|---|---|---|
| Chạy được? | Bước 1 không lỗi, có tool call, `n_tool_calls` > 0 | thử `VLLM_MAX_MODEL_LEN=16384`; nếu kernel không hỗ trợ T4 → cần GPU đời mới (Colab L4/A100) |
| Giống baseline? | `none` khớp công bố ≥ 9/12 | fp16/T4 lệch nhiều → kết quả chính nên chạy bf16 |
| Có tín hiệu? | `alignment` cứu được ≥ 3/8 task thất bại **và** hơn `bare`, giữ được 4/4 task pass | xem `gate_log` trong file JSON: model lặp lời gọi bị chặn hay bỏ cuộc? |

Tải `frameguard_runs.zip` (tab *Output*) và gửi lại để phân tích chi tiết (mỗi task một file JSON: hội thoại, tool call, quyết định chặn, tin nhắn phản hồi).